<a href="https://colab.research.google.com/github/SunJ2432/ai-image-compliance/blob/main/notebooks/01_generate_samples_paired.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Generate paired samples with SDXL-Turbo

Creates **three one-to-one AI-edited counterparts** from three uploaded real photographs, using prompts from `configs/sample_prompts.yaml`.

**Outputs:** six **256×256 RGB JPEG (quality 90)** images, `samples/manifest.csv`, `configs/sample_prompts.yaml`, and a downloadable `samples.zip`.

We center-crop both real and generated images to square to prevent aspect ratio from becoming a shortcut for the detector. SDXL-Turbo performs generation at **512×512**, then both classes are standardized to **256×256**.

> **Labels:** `real` and `ai_edited`. Img2img is AI editing, not proof that an image was created entirely from scratch. Six samples are a smoke test, not a reliable accuracy benchmark.


In [1]:
%pip install -q -U diffusers transformers accelerate safetensors Pillow PyYAML

## 1. Upload the three originals and `sample_prompts.yaml`

Download the provided `sample_prompts.yaml` and upload it together with your three original JPGs. The filenames must match the YAML entries. Select **Runtime → Change runtime type → T4 GPU** first.

If you already have these files in Colab, you may skip the upload and edit `input_dir` and `config_path` in the next cell.

In [3]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
input_dir = Path("/content")
config_path = input_dir / "sample_prompts.yaml"
assert config_path.is_file(), "Upload sample_prompts.yaml along with the images."
print("Uploaded:", list(uploaded))

Uploaded: []


In [4]:
import yaml, torch
from PIL import Image, ImageOps
from pathlib import Path

with open(config_path, encoding="utf-8") as f:
    config = yaml.safe_load(f)

assert torch.cuda.is_available(), "Enable a GPU runtime (T4 or better)."
assert len(config["samples"]) == 3

work = Path("/content/image_compliance_dataset")
real_dir = work / "samples" / "real"
ai_dir = work / "samples" / "ai_generated"
cfg_dir = work / "configs"
for folder in (real_dir, ai_dir, cfg_dir):
    folder.mkdir(parents=True, exist_ok=True)

import shutil
shutil.copy2(config_path, cfg_dir / "sample_prompts.yaml")

for sample in config["samples"]:
    assert (input_dir / sample["input_filename"]).is_file(), f"Missing: {sample['input_filename']}"
print("Inputs and configuration verified.")

Inputs and configuration verified.


## 2. Standardize the real images and prepare 512×512 img2img inputs

In [5]:
def standardize(img, size=256):
    # Crop without stretching, then resize. Both classes have identical geometry.
    img = ImageOps.exif_transpose(img).convert("RGB")
    return ImageOps.fit(img, (size, size), method=Image.Resampling.LANCZOS)

init_images = {}
for sample in config["samples"]:
    sid = sample["id"]
    with Image.open(input_dir / sample["input_filename"]) as im:
        init_images[sid] = standardize(im, 512)
        real = standardize(im, 256)
    real.save(real_dir / f"real_{sid}.jpg", "JPEG", quality=90, subsampling=0)
    print("Real:", sid, real.size)

Real: 001 (256, 256)
Real: 002 (256, 256)
Real: 003 (256, 256)


## 3. Load SDXL-Turbo locally on the Colab GPU

If you see an import error involving `torchao.quantization.FqnToConfig`, uninstall `torchao` (`%pip uninstall -y torchao`), restart the session, and rerun the cells. No Hugging Face token is normally needed for this public model.

In [6]:
from diffusers import AutoPipelineForImage2Image

pipe = AutoPipelineForImage2Image.from_pretrained(
    config["model"],
    torch_dtype=torch.float16,
    variant="fp16"
)
# CPU offload can help on a T4 with limited VRAM.
pipe.enable_model_cpu_offload()
print("SDXL-Turbo ready.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

SDXL-Turbo ready.


## 4. Generate one AI-edited counterpart per original

In [ ]:
settings = config["generation"]
for idx, sample in enumerate(config["samples"]):
    sid = sample["id"]
    generator = torch.Generator(device="cpu").manual_seed(settings["seed"] + idx)
    result = pipe(
        prompt=sample["prompt"],
        image=init_images[sid],
        num_inference_steps=settings["num_inference_steps"],
        strength=settings["strength"],
        guidance_scale=settings["guidance_scale"],
        generator=generator
    ).images[0]
    ai = standardize(result, 256)
    ai.save(ai_dir / f"ai_{sid}.jpg", "JPEG", quality=90, subsampling=0)
    print("Generated:", sid, ai.size)

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


## 5. Preview each pair

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 2, figsize=(8, 12))
for row, sample in enumerate(config["samples"]):
    sid = sample["id"]
    for col, path in enumerate([real_dir / f"real_{sid}.jpg", ai_dir / f"ai_{sid}.jpg"]):
        with Image.open(path) as im:
            axes[row, col].imshow(im)
        axes[row, col].set_title(f"{'Real' if col == 0 else 'AI-edited'} {sid}")
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

## 6. Build `samples/manifest.csv`, validate all images, and download `samples.zip`

In [ ]:
import csv, zipfile
manifest = work / "samples" / "manifest.csv"
rows = []
for sample in config["samples"]:
    sid = sample["id"]
    rows.append({
        "filename": f"real/real_{sid}.jpg",
        "label": "real",
        "source": f"uploaded_photo:{sample['input_filename']}",
        "prompt": ""
    })
    rows.append({
        "filename": f"ai_generated/ai_{sid}.jpg",
        "label": "ai_edited",
        "source": f"sdxl-turbo-img2img:real_{sid}.jpg",
        "prompt": sample["prompt"]
    })

with manifest.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["filename", "label", "source", "prompt"])
    writer.writeheader()
    writer.writerows(rows)

for row in rows:
    p = work / "samples" / row["filename"]
    with Image.open(p) as im:
        assert im.format == "JPEG" and im.mode == "RGB" and im.size == (256, 256), p
print("Verified six 256×256 RGB JPEG images (quality 90 at save time).")
print("Manifest:", manifest)

zip_path = Path("/content/samples.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(work.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(work))
print("Created:", zip_path)
files.download(str(zip_path))